## Example 1: Basic Redis Client Setup and Configuration

In [ ]:
# Example 1: Basic Redis Client Setup and Configuration
import redis
from redis import ConnectionPool
import time

print("=== Redis Python Client ===")

print(f"Redis library version: {redis.__version__}")

# Available components
components = [attr for attr in dir(redis) if not attr.startswith('_')]
print(f"Available components: {len(components)}")
print(f"Key components: {components[:12]}")

print(f"\n=== Client Configuration ===")

# Different ways to create Redis clients
clients = []

# Method 1: Direct instantiation
client1 = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
clients.append(("Direct", client1))

# Method 2: From URL
client2 = redis.from_url('redis://localhost:6379/1', decode_responses=True)
clients.append(("From URL", client2))

# Method 3: With connection pool
pool = ConnectionPool(host='localhost', port=6379, db=2, max_connections=10)
client3 = redis.Redis(connection_pool=pool, decode_responses=True)
clients.append(("With Pool", client3))

for name, client in clients:
    print(f"{name} client:")
    print(f"  Type: {type(client).__name__}")
    print(f"  Connection pool: {type(client.connection_pool).__name__}")
    
    # Get connection parameters
    conn_kwargs = client.connection_pool.connection_kwargs
    print(f"  Host: {conn_kwargs.get('host', 'N/A')}")
    print(f"  Port: {conn_kwargs.get('port', 'N/A')}")
    print(f"  DB: {conn_kwargs.get('db', 'N/A')}")
    print(f"  Decode responses: {conn_kwargs.get('decode_responses', False)}")

print(f"\n=== Connection Configuration Options ===")

# Various configuration options
config_options = {
    'socket_timeout': 5.0,
    'socket_connect_timeout': 5.0,
    'socket_keepalive': True,
    'socket_keepalive_options': {},
    'retry_on_timeout': True,
    'max_connections': 20,
    'health_check_interval': 30
}

configured_client = redis.Redis(
    host='localhost',
    port=6379,
    **config_options,
    decode_responses=True
)

print(f"Configured client created with options:")
configured_kwargs = configured_client.connection_pool.connection_kwargs
for key in config_options:
    if key == 'max_connections':  # a pool setting, not a connection setting
        actual_value = configured_client.connection_pool.max_connections
    else:
        actual_value = configured_kwargs.get(key, 'N/A')
    print(f"  {key}: {actual_value}")

# Creating a client does not connect; the settings are stored on its connection pool
assert [(c.connection_pool.connection_kwargs['host'], c.connection_pool.connection_kwargs['port'],
         c.connection_pool.connection_kwargs['db']) for _, c in clients] == [
    ('localhost', 6379, 0), ('localhost', 6379, 1), ('localhost', 6379, 2)]
assert client1.connection_pool.connection_kwargs['decode_responses'] is True
# A client built on an existing pool uses that pool's settings, so decode_responses=True is ignored here
assert client3.connection_pool is pool and 'decode_responses' not in pool.connection_kwargs
assert pool.max_connections == 10
assert configured_kwargs['socket_timeout'] == 5.0 and configured_kwargs['socket_connect_timeout'] == 5.0
assert configured_kwargs['health_check_interval'] == 30
assert configured_client.connection_pool.max_connections == 20

print("Basic Redis client setup completed!")

## Example 2: RESP command encoding and commands sent to a stand-in server

In [ ]:
# Example 2: RESP command encoding and commands sent to a stand-in server
import redis
from redis.connection import Connection, Encoder
import socketserver
import threading

print("=== RESP command encoding ===")

# Every command goes over the wire as a RESP array of bulk strings.
# pack_command() only encodes; this Connection never connects.
packed = b''.join(Connection(host='localhost', port=6379).pack_command('SET', 'user:1', 'Alice'))
assert packed == b'*3\r\n$3\r\nSET\r\n$6\r\nuser:1\r\n$5\r\nAlice\r\n', packed
print(f"SET user:1 Alice -> {packed!r}")

# The encoder turns each argument into bytes: str as UTF-8, int and float as their decimal text
encoder = Encoder(encoding='utf-8', encoding_errors='strict', decode_responses=False)
assert encoder.encode('héllo') == 'héllo'.encode('utf-8')
assert encoder.encode(42) == b'42' and encoder.encode(1.5) == b'1.5'
try:
    encoder.encode(None)
except redis.DataError:
    print("None is rejected with DataError")
else:
    raise AssertionError('None must not be encoded')

print(f"\n=== Pipelines ===")

# A pipeline queues commands on the client until execute()
client = redis.Redis(host='localhost', port=6379, decode_responses=True)
pipe = client.pipeline()
pipe.set('user:1:name', 'Alice').incrby('user:count', 1).lpush('notifications', 'Welcome Alice')
assert len(pipe) == 3
assert [args for args, _options in pipe.command_stack] == [
    ('SET', 'user:1:name', 'Alice'), ('INCRBY', 'user:count', 1), ('LPUSH', 'notifications', 'Welcome Alice')]
pipe.reset()
assert len(pipe) == 0
print("Pipeline queued 3 commands without sending them")

print(f"\n=== Commands sent to a stand-in server ===")


class StandInRedis(socketserver.StreamRequestHandler):
    """Answers a few commands with RESP2 replies and records every command it receives.

    It keeps data in a dict and implements only what these examples use; it is not a Redis server.
    """

    def read_command(self):
        line = self.rfile.readline()
        if not line:
            return None
        args = []
        for _ in range(int(line[1:])):
            size = int(self.rfile.readline()[1:])
            args.append(self.rfile.read(size + 2)[:-2])
        return args

    def handle(self):
        store = self.server.store
        while (args := self.read_command()) is not None:
            name = args[0].upper().decode()
            if name != 'CLIENT':  # the client announces itself with CLIENT SETINFO when it connects
                self.server.seen.append([a.decode('utf-8', 'replace') for a in args])
            if name in ('CLIENT', 'SET'):
                if name == 'SET':
                    store[args[1]] = args[2]
                reply = b'+OK\r\n'
            elif name == 'PING':
                reply = b'+PONG\r\n'
            elif name == 'GET':
                value = store.get(args[1])
                reply = b'$-1\r\n' if value is None else b'$%d\r\n%s\r\n' % (len(value), value)
            elif name == 'INCRBY':
                store[args[1]] = b'%d' % (int(store.get(args[1], b'0')) + int(args[2]))
                reply = b':%s\r\n' % store[args[1]]
            elif name == 'PUBLISH':
                reply = b':0\r\n'  # number of subscribers that received the message
            elif name == 'XADD':
                self.server.stream_ids += 1
                entry_id = b'%d-0' % self.server.stream_ids
                reply = b'$%d\r\n%s\r\n' % (len(entry_id), entry_id)
            else:
                reply = b"-ERR unknown command '%s'\r\n" % args[0]
            self.wfile.write(reply)


server = socketserver.ThreadingTCPServer(('127.0.0.1', 0), StandInRedis)
server.daemon_threads = True
server.store, server.seen, server.stream_ids = {}, [], 0
threading.Thread(target=server.serve_forever, daemon=True).start()
host, port = server.server_address

# protocol=2 selects RESP2, which the stand-in speaks
pool = redis.ConnectionPool(host=host, port=port, protocol=2, decode_responses=True)
client = redis.Redis(connection_pool=pool)

assert client.ping() is True
assert client.set('user:1', 'Alice') is True
assert client.get('user:1') == 'Alice'
assert client.get('missing') is None
assert client.incr('counter') == 1 and client.incrby('counter', 5) == 6
pipe = client.pipeline(transaction=False)
pipe.set('a', '1').get('a').incr('counter')
assert pipe.execute() == [True, '1', 7]
# The client sent exactly these commands; INCR without an amount is sent as INCRBY 1
assert server.seen == [
    ['PING'], ['SET', 'user:1', 'Alice'], ['GET', 'user:1'], ['GET', 'missing'],
    ['INCRBY', 'counter', '1'], ['INCRBY', 'counter', '5'],
    ['SET', 'a', '1'], ['GET', 'a'], ['INCRBY', 'counter', '1'],
], server.seen
print(f"Stand-in server on port {port} received {len(server.seen)} commands")

## Example 3: Connection Pool Management and Error Handling

In [ ]:
# Example 3: Connection Pool Management and Error Handling
import redis
from redis import ConnectionPool, ConnectionError, TimeoutError, ResponseError
import time

print("=== Connection Pool Management ===")

# Connection pool configurations
pool_configs = [
    {
        'name': 'Basic Pool',
        'max_connections': 10,
        'host': 'localhost',
        'port': 6379,
        'db': 0
    },
    {
        'name': 'High Capacity Pool',
        'max_connections': 50,
        'host': 'localhost', 
        'port': 6379,
        'db': 0,
        'socket_timeout': 10,
        'socket_connect_timeout': 10,
        'retry_on_timeout': True
    },
    {
        'name': 'SSL Pool',
        'max_connections': 20,
        'host': 'secure-redis.example.com',
        'port': 6380,
        # A pool takes TLS as its connection class. An `ssl=True` kwarg would be passed on to a plain
        # Connection, which rejects it with TypeError the first time the pool creates a connection.
        'connection_class': redis.SSLConnection,
        'ssl_cert_reqs': None
    }
]

pools = []
for config in pool_configs:
    name = config.pop('name')
    pool = ConnectionPool(**config)
    pools.append((name, pool))
    print(f"{name} created:")
    print(f"  Max connections: {pool.max_connections}")
    print(f"  Host: {pool.connection_kwargs.get('host')}")
    print(f"  Port: {pool.connection_kwargs.get('port')}")
    print(f"  SSL: {issubclass(pool.connection_class, redis.SSLConnection)}")
    
    # Create client from pool
    client = redis.Redis(connection_pool=pool)
    print(f"  Client created: {type(client).__name__}")
    # Construction is lazy (no server contact). make_connection() builds a connection object without
    # connecting, which is where a setting the connection class does not accept would raise.
    connection = pool.make_connection()
    assert type(connection) is pool.connection_class and client.connection_pool is pool, (connection, pool)
    assert (connection.host, connection.port) == (config['host'], config['port']), connection
    assert pool.max_connections == config['max_connections'], pool
    connection.disconnect()
    

assert [(name, pool.connection_class.__name__) for name, pool in pools] == [
    ('Basic Pool', 'Connection'), ('High Capacity Pool', 'Connection'), ('SSL Pool', 'SSLConnection')], pools

print(f"\n=== Error Handling ===")

# Every redis-py exception derives from RedisError. AuthenticationError is a kind of ConnectionError,
# so an `except` for it must come before the one for ConnectionError.
for exc_class in (redis.ConnectionError, redis.TimeoutError, redis.ResponseError, redis.DataError,
                  redis.AuthenticationError):
    assert issubclass(exc_class, redis.RedisError), exc_class
    print(f"  {exc_class.__name__}: bases {[cls.__name__ for cls in exc_class.__bases__]}")
assert issubclass(redis.AuthenticationError, redis.ConnectionError)


def classify(operation):
    """Run a Redis call and name the kind of failure, if any."""
    try:
        operation()
        return 'ok'
    except redis.AuthenticationError:
        return 'authentication'
    except redis.ConnectionError:
        return 'connection'
    except redis.TimeoutError:
        return 'timeout'
    except redis.ResponseError:
        return 'response'


# A port that nothing listens on: bind a socket to get a free port, then close it
import socket
from redis.backoff import NoBackoff
from redis.retry import Retry

with socket.socket() as probe:
    probe.bind(('127.0.0.1', 0))
    closed_port = probe.getsockname()[1]
# Retry(NoBackoff(), 0) fails at once instead of retrying the connection
unreachable = redis.Redis(host='127.0.0.1', port=closed_port, socket_connect_timeout=1, retry=Retry(NoBackoff(), 0))
# Uses the stand-in server from Example 2
reachable = redis.Redis(host=host, port=port, protocol=2, decode_responses=True)

results = {
    'ping': classify(reachable.ping),
    'refused': classify(unreachable.ping),
    'unknown command': classify(lambda: reachable.execute_command('NOSUCHCOMMAND')),
}
print(f"Results: {results}")
assert results == {'ping': 'ok', 'refused': 'connection', 'unknown command': 'response'}, results

try:
    reachable.execute_command('NOSUCHCOMMAND')
except redis.ResponseError as error:
    assert "unknown command 'NOSUCHCOMMAND'" in str(error), error

print(f"\n=== Connection Health Check ===")


def redis_health_check(client):
    """True when the server answers PING."""
    try:
        return client.ping()
    except (redis.ConnectionError, redis.TimeoutError):
        return False


assert redis_health_check(reachable) is True
assert redis_health_check(unreachable) is False
print("Health check: stand-in server PASS, closed port FAIL")

## Example 4: Pub/Sub, Lua scripts and streams

In [ ]:
# Example 4: Pub/Sub, Lua scripts and streams
# Uses the stand-in server (host, port, server) from Example 2.
import redis
import hashlib
import json

client = redis.Redis(host=host, port=port, protocol=2, decode_responses=True)

print("=== Pub/Sub ===")

# A PubSub object holds subscriptions; it opens its own connection on the first subscribe
pubsub = client.pubsub()
assert pubsub.channels == {} and pubsub.patterns == {} and not pubsub.subscribed
pubsub.close()

# PUBLISH returns how many subscribers received the message; there are none here
message = json.dumps({'user_id': 123, 'action': 'login'})
assert client.publish('user_events', message) == 0
assert server.seen[-1] == ['PUBLISH', 'user_events', message], server.seen[-1]
print(f"PUBLISH user_events -> 0 receivers")

print(f"\n=== Lua scripts ===")

# register_script() computes the SHA1 that EVALSHA uses; nothing is sent until the script is called
lua = 'local current = redis.call("incr", KEYS[1]); if current == 1 then redis.call("expire", KEYS[1], ARGV[1]) end; return current'
rate_limiter = client.register_script(lua)
assert rate_limiter.sha == hashlib.sha1(lua.encode()).hexdigest(), rate_limiter.sha
print(f"Rate limiter script SHA1: {rate_limiter.sha}")

print(f"\n=== Streams ===")

# XADD sends the fields as name/value pairs; "*" lets the server choose the entry id
first_id = client.xadd('events', {'user_id': '123', 'action': 'click', 'page': 'home'})
assert server.seen[-1] == ['XADD', 'events', '*', 'user_id', '123', 'action', 'click', 'page', 'home'], server.seen[-1]
# maxlen with approximate=True trims the stream to about that many entries
second_id = client.xadd('events', {'user_id': '456', 'action': 'purchase'}, maxlen=1000, approximate=True)
assert server.seen[-1] == ['XADD', 'events', 'MAXLEN', '~', '1000', '*', 'user_id', '456', 'action', 'purchase'], server.seen[-1]
assert (first_id, second_id) == ('1-0', '2-0'), (first_id, second_id)
print(f"XADD returned entry ids {first_id} and {second_id}")

## Example 5: Redis Data Serialization and Caching Patterns

In [ ]:
# Example 5: Redis Data Serialization and Caching Patterns
# Uses the stand-in server (host, port, server) from Example 2.
import redis
import json
import pickle

print("=== Redis Data Serialization ===")

# decode_responses=False returns bytes; decode_responses=True returns str
client = redis.Redis(host=host, port=port, protocol=2, decode_responses=False)
client_text = redis.Redis(host=host, port=port, protocol=2, decode_responses=True)

data_samples = {
    'user_profile': {
        'id': 12345,
        'name': 'Alice Johnson',
        'email': 'alice@example.com',
        'preferences': {'theme': 'dark', 'notifications': True},
        'created_at': '2024-01-15T10:30:00',
    },
    'product_catalog': [
        {'id': 1, 'name': 'Widget A', 'price': 29.99, 'stock': 100},
        {'id': 2, 'name': 'Widget B', 'price': 39.99, 'stock': 50},
    ],
    'session_data': {
        'user_id': 12345,
        'login_time': 1705314600.0,
        'permissions': ['read', 'write', 'admin'],
    },
}

# JSON is text, so the text client reads it back as str
for key, data in data_samples.items():
    client_text.set(f'json:{key}', json.dumps(data))
    stored = client_text.get(f'json:{key}')
    assert isinstance(stored, str) and json.loads(stored) == data, key
    print(f"  JSON {key}: {len(stored)} chars")

# Pickle is binary, so it needs the bytes client
for key, data in data_samples.items():
    client.set(f'pickle:{key}', pickle.dumps(data))
    stored = client.get(f'pickle:{key}')
    assert isinstance(stored, bytes) and pickle.loads(stored) == data, key
    print(f"  Pickle {key}: {len(stored)} bytes")

print(f"\n=== Cache-Aside Pattern ===")

database_reads = []


def load_profile_from_database(user_id):
    database_reads.append(user_id)
    return {'id': user_id, 'name': 'Alice Johnson'}


def get_profile(user_id, ttl=3600):
    """Read from the cache; on a miss, load from the database and cache the result with a TTL."""
    key = f'user:{user_id}:profile'
    cached = client_text.get(key)
    if cached is not None:
        return json.loads(cached)
    profile = load_profile_from_database(user_id)
    client_text.set(key, json.dumps(profile), ex=ttl)
    return profile


first = get_profile(12345)
second = get_profile(12345)
assert first == second == {'id': 12345, 'name': 'Alice Johnson'}
assert database_reads == [12345], database_reads  # the second call was served from the cache
# The TTL travels with SET as "EX <seconds>"
assert ['SET', 'user:12345:profile', json.dumps(first), 'EX', '3600'] in server.seen
print(f"Two reads, {len(database_reads)} database load")

client.close()
client_text.close()
server.shutdown()
server.server_close()
print("Stand-in server stopped")